# SQL_Data_Base

# 5. Connect_the_SQL_Data_Base

## Store and validate (2 marks). 

1. Write the three clean tables to data/clean/ as CSV files and to the SQLite database data/instamart.db as tables named customers, products, orders (for example with sqlite3.connect and DataFrame.to_sql, or an equivalent method; store order_ts as text in the form YYYY-MM-DD HH:MM:SS). 

2. Write a validation script that asserts at least eight of the reference checkpoints from tasks 3 and 4 and prints exactly ALL CHECKPOINTS PASSED when all hold.

In [2]:
import pandas as pd

In [3]:
order_clean = pd.read_csv("data/clean/Order_details_cleaned.csv")

In [4]:
order_db = order_clean.copy()
order_db

,order_id,customer_id,product_id,quantity,order_ts,status,discount_pct,order_amount,delivery_minutes
0,ORD-110093,C0967,P020,1,2025-04-01 07:07:28,pending,15,127.50,NaN
1,ORD-110093,C0967,P020,1,2025-04-01 07:08:28,completed,15,127.50,19.0
2,ORD-111396,C1117,P036,2,2025-04-01 07:11:59,completed,0,1598.00,25.0
3,ORD-102006,C0194,P009,1,2025-04-01 07:12:43,cancelled,25,21.75,NaN
4,ORD-105921,C0569,P005,2,2025-04-01 07:54:43,completed,10,86.40,33.0
...,...,...,...,...,...,...,...,...,...
12954,ORD-103085,C0289,P002,3,2025-09-30 21:04:57,completed,15,420.75,18.0
12955,ORD-112223,C1195,P019,2,2025-09-30 21:27:46,completed,0,60.00,25.0
12956,ORD-100182,C0021,P034,2,2025-09-30 21:37:26,completed,15,2208.30,25.0
12957,ORD-108027,C0767,P009,2,2025-09-30 22:30:20,completed,0,58.00,29.0


In [5]:
order_db["order_ts"] = pd.to_datetime(order_db["order_ts"], errors = "coerce")

#order_db["order_ts"]=
#print(order_db["order_ts"].dt.strftime("%Y-%M-%D %H:%M:%S"))

In [6]:
customer_clean = pd.read_csv("data/clean/Customer_data_Cleaned.csv")

customer_data = customer_clean.copy()

In [7]:
product_clean = pd.read_csv("data/clean/Product_Cleaned.csv")

product_data = product_clean.copy()

In [8]:
import sqlite3

## 1. Create_Data_Base

In [9]:
db_path = "data/instamart.db" # create a path.

conn = sqlite3.connect(db_path) # connect a path to the database.

print("Data base Created and connected successfully.")

Data base Created and connected successfully.


## 2. Connect_to_the_database(product,customer,order_db)

In [10]:
customer_data.to_sql(
                        "customers",
                        conn,
                        if_exists="replace",
                        index=False
)

product_data.to_sql(
                        "products",
                        conn,
                        if_exists="replace",
                        index=False
)

order_db.to_sql(
                        "orders",
                        conn,
                        if_exists="replace",
                        index=False
)

print("All Tables are stored Successfully.")

All Tables are stored Successfully.


In [11]:
tables = pd.read_sql_query(
            """
            SELECT name
            FROM sqlite_master
            WHERE type = "table";
            """,
            conn
            
        )

display(tables)
print(tables)

,name
0,customers
1,products
2,orders


        name
0  customers
1   products
2     orders


In [12]:
print("Customes_Rows : " , pd.read_sql_query(
                    "SELECT COUNT(*) AS total FROM customers",conn
                    ).iloc[0,0])

print("products_Rows : " , pd.read_sql_query(
                    "SELECT COUNT(*) AS total FROM products",conn
                    ).iloc[0,0])

print("orders_Rows : " , pd.read_sql_query(
                    "SELECT COUNT(*) AS total FROM orders",conn
                    ).iloc[0,0])

Customes_Rows :  1200
products_Rows :  40
orders_Rows :  12959


In [13]:
pd.read_sql_query(
                    "SELECT * FROM orders LIMIT 5 ;",
                    conn
)

,order_id,customer_id,product_id,quantity,order_ts,status,discount_pct,order_amount,delivery_minutes
0,ORD-110093,C0967,P020,1,2025-04-01 07:07:28,pending,15,127.50,NaN
1,ORD-110093,C0967,P020,1,2025-04-01 07:08:28,completed,15,127.50,19.0
2,ORD-111396,C1117,P036,2,2025-04-01 07:11:59,completed,0,1598.00,25.0
3,ORD-102006,C0194,P009,1,2025-04-01 07:12:43,cancelled,25,21.75,NaN
4,ORD-105921,C0569,P005,2,2025-04-01 07:54:43,completed,10,86.40,33.0


In [14]:
pd.read_sql_query(
                "SELECT order_ts FROM orders LIMIT 5 ; ",
                conn
)

,order_ts
0,2025-04-01 07:07:28
1,2025-04-01 07:08:28
2,2025-04-01 07:11:59
3,2025-04-01 07:12:43
4,2025-04-01 07:54:43


## 6. SQL analysis (5 marks). Save seven queries in sql/queries.sql, run each from Python with pd.read_sql_query against data/instamart.db, and report the stated result for each. Unless stated otherwise, only completed orders count as revenue or spend.

Q1 — WHERE then HAVING. Keep only completed orders with order_amount > 500 (WHERE), group by customer, and keep customers whose total of those orders exceeds 8000 (HAVING). Reference: 30 customers.

Q2 — three-table join. Revenue by city and product category (join orders, customers, products). Reference: 30 rows; the 30 revenues add up to INR 2,998,668.80; the largest cell is Bengaluru / Electronics at INR 233,909.45.

Q3 — scalar subquery. Products priced above the average catalogue price. Reference: 11 products.

Q4 — LEFT JOIN. Customers who have no completed order. Reference: 25 customers.

Q5 — common table expression. One CTE computes each customer's total completed spend and a second computes the average of those totals; the main query 
returns customers whose total is above the average. Reference: 550 customers; average INR 2,552.06 (rounded to 2 decimals).

Q6 — ranking window functions. For each customer with completed spend, compute total spend and then RANK(), DENSE_RANK() and ROW_NUMBER() using OVER (PARTITION BY city ORDER BY total_spend DESC). Reference: the top customer per city is Bengaluru C0082 (12,856.20), Chennai C0930 (10,151.40), Delhi C0291 (13,742.60), Hyderabad C0743 (12,730.05), Mumbai C0693 (14,766.55), Pune C0426 (10,377.10); exactly 8 rows have RANK() different from ROW_NUMBER(), and your README states in one sentence why ties cause that.

Q7 — running total. Daily completed revenue (the day is the first ten characters of order_ts) with a cumulative revenue column computed as SUM(revenue) OVER (ORDER BY day). Reference: 183 daily rows; the last cumulative value is INR 2,998,668.80.

In [42]:
order_db

,order_id,customer_id,product_id,quantity,order_ts,status,discount_pct,order_amount,delivery_minutes
0,ORD-110093,C0967,P020,1,2025-04-01 07:07:28,pending,15,127.50,NaN
1,ORD-110093,C0967,P020,1,2025-04-01 07:08:28,completed,15,127.50,19.0
2,ORD-111396,C1117,P036,2,2025-04-01 07:11:59,completed,0,1598.00,25.0
3,ORD-102006,C0194,P009,1,2025-04-01 07:12:43,cancelled,25,21.75,NaN
4,ORD-105921,C0569,P005,2,2025-04-01 07:54:43,completed,10,86.40,33.0
...,...,...,...,...,...,...,...,...,...
12954,ORD-103085,C0289,P002,3,2025-09-30 21:04:57,completed,15,420.75,18.0
12955,ORD-112223,C1195,P019,2,2025-09-30 21:27:46,completed,0,60.00,25.0
12956,ORD-100182,C0021,P034,2,2025-09-30 21:37:26,completed,15,2208.30,25.0
12957,ORD-108027,C0767,P009,2,2025-09-30 22:30:20,completed,0,58.00,29.0


In [26]:
# Q_1 _ WHERE then HAVING

query_1 = pd.read_sql_query(
                            """
                            SELECT customer_id, SUM(order_amount) AS spend_over_500
                            FROM orders WHERE status='completed' AND order_amount > 500
                            GROUP BY customer_id HAVING SUM(order_amount) > 8000;
                            """,
                            conn
        )

print(query_1)
# Q2 — three-table join.

query_2 = pd.read_sql_query(
                            """
                            SELECT c.city, p.category, ROUND(SUM(o.order_amount),2) AS revenue
                            FROM orders o JOIN customers c USING(customer_id) JOIN products p USING(product_id)
                            WHERE o.status='completed'
                            GROUP BY c.city,p.category ORDER BY c.city,p.category;
                            """,
                            conn
            )

print(query_2)

# Q3: products above average catalogue price

query_3 = pd.read_sql_query(
                            """
                            SELECT product_id, product_name, price_inr FROM products
                            WHERE price_inr > (SELECT AVG(price_inr) FROM products);
                            """,
                            conn
            )

print(query_3)

# Q4: customers without completed orders

query_4 = pd.read_sql_query(
                        """
                        SELECT c.customer_id FROM customers c LEFT JOIN orders o
                        ON c.customer_id=o.customer_id AND o.status='completed'
                        WHERE o.order_id IS NULL;
                        """,
                        conn
        )


# -- Q5: CTE for above-average customer spend


query_5 = pd.read_sql_query(
                            """
                            WITH customer_spend AS (
                             SELECT customer_id, SUM(order_amount) AS total_spend FROM orders
                             WHERE status='completed' GROUP BY customer_id
                            ), average_spend AS (SELECT AVG(total_spend) AS avg_spend FROM customer_spend)
                            SELECT customer_id,total_spend FROM customer_spend,average_spend
                            WHERE total_spend > avg_spend;
                            """,
                            conn
        )

print(query_5)

# -- Q6: ranking window functions within city

query_6 = pd.read_sql_query(
                            """
                            WITH spend AS (
                             SELECT c.customer_id,c.city,SUM(o.order_amount) AS total_spend
                             FROM customers c JOIN orders o USING(customer_id)
                             WHERE o.status='completed' GROUP BY c.customer_id,c.city
                            )
                            SELECT *, RANK() OVER(PARTITION BY city ORDER BY total_spend DESC) AS spend_rank,
                             DENSE_RANK() OVER(PARTITION BY city ORDER BY total_spend DESC) AS dense_rank,
                             ROW_NUMBER() OVER(PARTITION BY city ORDER BY total_spend DESC) AS row_num
                            FROM spend;
                            """,
                            conn
        )

print(query_6)

# -- Q7: daily running total

query_7 = pd.read_sql_query(
                            """
                            WITH daily AS (
                             SELECT substr(order_ts,1,10) AS day,SUM(order_amount) AS revenue
                             FROM orders WHERE status='completed' GROUP BY substr(order_ts,1,10)
                            )
                            SELECT day,revenue,SUM(revenue) OVER(ORDER BY day ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cumulative_revenue
                            FROM daily ORDER BY day;
                            """,
                            conn
        )

display(query_7)

   customer_id  spend_over_500
0        C0082        12317.70
1        C0130         9170.40
2        C0147         9141.50
3        C0167         8718.55
4        C0267        12150.25
5        C0291        12150.70
6        C0308         9872.40
7        C0332        10779.50
8        C0407         9451.75
9        C0424         8294.20
10       C0426         9884.60
11       C0454         8299.15
12       C0488        11825.50
13       C0512        11617.65
14       C0525        12241.30
15       C0543        10636.10
16       C0693        13776.55
17       C0696         8941.75
18       C0699        11693.90
19       C0743        12266.85
20       C0775        10263.55
21       C0819         8695.30
22       C0916         8395.35
23       C0930         9194.00
24       C0942         8643.60
25       C1000         8462.20
26       C1024        10160.10
27       C1136         8983.75
28       C1146         8993.05
29       C1147         9740.80
         city       category    revenue

,day,revenue,cumulative_revenue
0,2025-04-01,16876.50,16876.50
1,2025-04-02,19854.55,36731.05
2,2025-04-03,28048.35,64779.40
3,2025-04-04,14249.60,79029.00
4,2025-04-05,16031.50,95060.50
...,...,...,...
178,2025-09-26,17190.85,2941590.45
179,2025-09-27,13689.95,2955280.40
180,2025-09-28,12689.85,2967970.25
181,2025-09-29,14022.15,2981992.40


In [44]:
# Q1
q1 = pd.read_sql_query(
    """
    SELECT
        customer_id,
        ROUND(SUM(order_amount), 2) AS spend_over_500
    FROM orders
    WHERE status = 'completed'
      AND order_amount > 500
    GROUP BY customer_id
    HAVING SUM(order_amount) > 8000
    ORDER BY spend_over_500 DESC;
    """,
    conn
)

display(q1)
print("Total qualifying customers:", len(q1))

,customer_id,spend_over_500
0,C0693,13776.55
1,C0082,12317.70
2,C0743,12266.85
3,C0525,12241.30
4,C0291,12150.70
5,C0267,12150.25
6,C0488,11825.50
7,C0699,11693.90
8,C0512,11617.65
9,C0332,10779.50


Total qualifying customers: 30


In [42]:
# Q2
q2 = pd.read_sql_query(
    """
    SELECT
        c.city,
        p.category,
        ROUND(SUM(o.order_amount), 2) AS revenue
    FROM orders AS o
    JOIN customers AS c
        ON o.customer_id = c.customer_id
    JOIN products AS p
        ON o.product_id = p.product_id
    WHERE o.status = 'completed'
    GROUP BY c.city, p.category
    ORDER BY c.city, p.category;
    """,
    conn
)

display(q2)
print("Total city-category combinations:", len(q2))
print("Total revenue: INR", round(q2["revenue"].sum(), 2))

,city,category,revenue
0,Bengaluru,Dairy,86988.05
1,Bengaluru,Electronics,233909.45
2,Bengaluru,Grocery,110915.65
3,Bengaluru,Personal Care,20193.85
4,Bengaluru,Snacks,71144.40
5,Chennai,Dairy,78301.80
6,Chennai,Electronics,195508.80
7,Chennai,Grocery,93817.45
8,Chennai,Personal Care,27675.40
9,Chennai,Snacks,68325.70


Total city-category combinations: 30
Total revenue: INR 2997495.5


In [43]:
# Q3 
q3 = pd.read_sql_query(
    """
    SELECT
        product_id,
        product_name,
        price_inr
    FROM products
    WHERE price_inr > (
        SELECT AVG(price_inr)
        FROM products
    )
    ORDER BY price_inr DESC;
    """,
    conn
)

display(q3)
print("Products above average price:", len(q3))

,product_id,product_name,price_inr
0,P038,Bluetooth Speaker,1999
1,P033,Wireless Earbuds,1499
2,P034,Power Bank 10000mAh,1299
3,P036,Phone Charger 20W,799
4,P037,Smart Bulb,599
5,P040,Memory Card 64GB,549
6,P032,Sunscreen 50g,399
7,P030,Body Lotion 400ml,350
8,P039,Phone Stand,349
9,P014,Ghee 500ml,340


Products above average price: 11


In [38]:
# Q4: customers without completed orders

q4 = pd.read_sql_query(
    """
    SELECT
        c.customer_id
    FROM customers AS c
    LEFT JOIN orders AS o
        ON c.customer_id = o.customer_id
        AND o.status = 'completed'
    WHERE o.order_id IS NULL
    ORDER BY c.customer_id;
    """,
    conn
)

display(q4)
print("Customers without completed orders:", len(q4))

,customer_id
0,C0029
1,C0037
2,C0093
3,C0115
4,C0120
5,C0151
6,C0354
7,C0363
8,C0412
9,C0461


Customers without completed orders: 25


In [37]:
# -- Q5: CTE for above-average customer spend

q5 = pd.read_sql_query(
    """
    WITH customer_spend AS (
        SELECT
            customer_id,
            SUM(order_amount) AS total_spend
        FROM orders
        WHERE status = 'completed'
        GROUP BY customer_id
    ),
    average_spend AS (
        SELECT
            AVG(total_spend) AS avg_spend
        FROM customer_spend
    )
    SELECT
        customer_id,
        ROUND(total_spend, 2) AS total_spend
    FROM customer_spend
    CROSS JOIN average_spend
    WHERE total_spend > avg_spend
    ORDER BY total_spend DESC;
    """,
    conn
)

display(q5)
print("Above-average customers:", len(q5))

,customer_id,total_spend
0,C0693,14766.55
1,C0291,13742.60
2,C0488,13187.50
3,C0082,12856.20
4,C0743,12730.05
...,...,...
545,C0739,2570.30
546,C0272,2569.90
547,C0232,2569.60
548,C0987,2567.95


Above-average customers: 550


In [36]:
# -- Q6: ranking window functions within city

q6 = pd.read_sql_query(
    """
    WITH spend AS (
        SELECT
            c.customer_id,
            c.city,
            SUM(o.order_amount) AS total_spend
        FROM customers AS c
        JOIN orders AS o
            ON c.customer_id = o.customer_id
        WHERE o.status = 'completed'
        GROUP BY c.customer_id, c.city
    )
    SELECT
        customer_id,
        city,
        ROUND(total_spend, 2) AS total_spend,

        RANK() OVER (
            PARTITION BY city
            ORDER BY total_spend DESC
        ) AS spend_rank,

        DENSE_RANK() OVER (
            PARTITION BY city
            ORDER BY total_spend DESC
        ) AS dense_rank,

        ROW_NUMBER() OVER (
            PARTITION BY city
            ORDER BY total_spend DESC, customer_id
        ) AS row_num

    FROM spend
    ORDER BY city, spend_rank, customer_id;
    """,
    conn
)

display(q6)

print(
    "Rows where RANK differs from ROW_NUMBER:",
    (q6["spend_rank"] != q6["row_num"]).sum()
)

,customer_id,city,total_spend,spend_rank,dense_rank,row_num
0,C0082,Bengaluru,12856.20,1,1,1
1,C0525,Bengaluru,12719.70,2,2,2
2,C0699,Bengaluru,12471.80,3,3,3
3,C0543,Bengaluru,11195.10,4,4,4
4,C1147,Bengaluru,10156.90,5,5,5
...,...,...,...,...,...,...
1170,C0735,Pune,39.25,217,213,217
1171,C0541,Pune,38.25,218,214,218
1172,C1061,Pune,34.80,219,215,219
1173,C1158,Pune,33.60,220,216,220


Rows where RANK differs from ROW_NUMBER: 8


In [34]:
# -- Q7: daily running total

q7 = pd.read_sql_query(
    """
    WITH daily AS (
        SELECT
            substr(order_ts, 1, 10) AS day,
            SUM(order_amount) AS revenue
        FROM orders
        WHERE status = 'completed'
        GROUP BY substr(order_ts, 1, 10)
    )
    SELECT
        day,
        ROUND(revenue, 2) AS revenue,
        ROUND(
            SUM(revenue) OVER (
                ORDER BY day
                ROWS BETWEEN UNBOUNDED PRECEDING
                AND CURRENT ROW
            ),
            2
        ) AS cumulative_revenue
    FROM daily
    ORDER BY day;
    """,
    conn
)

display(q7)

print("Daily records:", len(q7))
print(
    "Final cumulative revenue: INR",
    round(q7["cumulative_revenue"].iloc[-1], 2)
)

,day,revenue,cumulative_revenue
0,2025-04-01,16876.50,16876.50
1,2025-04-02,19854.55,36731.05
2,2025-04-03,28048.35,64779.40
3,2025-04-04,14249.60,79029.00
4,2025-04-05,16031.50,95060.50
...,...,...,...
178,2025-09-26,17190.85,2941590.45
179,2025-09-27,13689.95,2955280.40
180,2025-09-28,12689.85,2967970.25
181,2025-09-29,14022.15,2981992.40


Daily records: 183
Final cumulative revenue: INR 2997495.5


# 7. pandas cross-check and exploratory analysis (2 marks). 

1. (a) Rebuild Q2 with pd.pivot_table(..., margins=True) and show that its grand total equals the SQL total; with groupby and named aggregation report, per city, the sum, mean and count of completed order_amount (reference: Bengaluru sum 523,643.40, mean 276.47, count 1,894; Mumbai sum 479,058.50, mean 297.00, count 1,613).
2. (b) For completed order_amount, report mean, median, skewness, the quartiles, the 1.5×IQR fences and the number of outliers beyond each fence, then answer in your own words, in at most 200 words in total,
3. the four questions:


  3.1. What is this data about?

  3.2. What do its distributions and statistics look like?
  
  3.3. Are there outliers or anomalies? What relationships exist between variables?
  
  3.4. For the last question use the correlation matrix of order_amount, discount_pct, quantity, delivery_minutes over completed orders and name the pair with the largest correlation in absolute value. Reference: mean 269.91 versus median 135.00 (right-skewed), skewness about 4.61, Q1 64.80, Q3 297.00, upper fence 645.30 with 897 outliers above it and none below; the correlation largest in absolute value is order_amount with discount_pct at about −0.095 (the pair order_amount and quantity is +0.087, slightly smaller in absolute value), and your answer must say that correlation of this size does not establish causation.

### Total revenue depand upon the catogery and city wise

In [45]:
completed_orders = order_clean[
    order_clean["status"] == "completed"
]

orders_with_city = completed_orders.merge(
    customer_clean[["customer_id", "city"]],
    on="customer_id",
    how="left"
)

orders_with_city = orders_with_city.merge(
    product_clean[["product_id", "category"]],
    on="product_id",
    how="left"
)

city_category_revenue = pd.pivot_table(
    orders_with_city,
    values="order_amount",
    index="city",
    columns="category",
    aggfunc="sum",
    fill_value=0,
    margins=True,
    margins_name="Total"
)

display(city_category_revenue.round(2))

category,Dairy,Electronics,Grocery,Personal Care,Snacks,Total
city,,,,,,
Bengaluru,86988.05,233909.45,110915.65,20193.85,71144.40,523151.40
Chennai,78301.80,195508.80,93817.45,27675.40,68325.70,463629.15
Delhi,117271.70,158549.10,112658.05,23344.00,90879.45,502702.30
Hyderabad,83536.00,228913.10,109475.40,24800.05,81670.35,528394.90
Mumbai,75209.85,214738.40,78970.15,38018.00,72122.10,479058.50
Pune,90349.55,187270.45,108456.60,34275.15,80207.50,500559.25
Total,531656.95,1218889.30,614293.30,168306.45,464349.50,2997495.50


### Customer_spending_by_city

In [46]:
city_spending = (
    orders_with_city
    .groupby("city")["order_amount"]
    .agg(
        total_spend="sum",
        average_spend="mean",
        order_count="count"
    )
    .round(2)
    .reset_index()
)

display(city_spending)

,city,total_spend,average_spend,order_count
0,Bengaluru,523151.40,276.51,1892
1,Chennai,463629.15,272.72,1700
2,Delhi,502702.30,243.68,2063
3,Hyderabad,528394.90,285.00,1854
4,Mumbai,479058.50,297.00,1613
5,Pune,500559.25,252.43,1983


In [47]:
amount_stats = completed_orders["order_amount"].describe()

display(amount_stats)

count    11105.000000
mean       269.923053
std        432.157484
min         12.000000
25%         64.800000
50%        135.000000
75%        297.000000
max       3998.000000
Name: order_amount, dtype: float64

In [48]:
Q1 = completed_orders["order_amount"].quantile(0.25)
Q3 = completed_orders["order_amount"].quantile(0.75)

IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

outliers = completed_orders[
    (completed_orders["order_amount"] < lower_limit) |
    (completed_orders["order_amount"] > upper_limit)
]

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower limit:", lower_limit)
print("Upper limit:", upper_limit)
print("Total outliers:", len(outliers))

Q1: 64.8
Q3: 297.0
IQR: 232.2
Lower limit: -283.49999999999994
Upper limit: 645.3
Total outliers: 897


In [49]:
correlation_data = completed_orders[
    ["order_amount", "quantity", "discount_pct", "delivery_minutes"]
].corr(numeric_only=True)

display(correlation_data.round(3))

,order_amount,quantity,discount_pct,delivery_minutes
order_amount,1.000,0.087,-0.095,-0.003
quantity,0.087,1.000,-0.000,-0.012
discount_pct,-0.095,-0.000,1.000,-0.010
delivery_minutes,-0.003,-0.012,-0.010,1.000


## Extract correlations with order amount

In [51]:
order_amount_corr = correlation_data["order_amount"].drop("order_amount")

display(order_amount_corr.sort_values(ascending=False))

quantity            0.087241
delivery_minutes   -0.003191
discount_pct       -0.095112
Name: order_amount, dtype: float64

## Identify the strongest relationship

In [52]:
strongest_corr = order_amount_corr.abs().idxmax()

print("Strongest correlation variable:", strongest_corr)

print(
    "Correlation value:",
    round(correlation_data.loc["order_amount", strongest_corr], 3)
)

Strongest correlation variable: discount_pct
Correlation value: -0.095


# Discriptive

### Exploratory Data Analysis and Interpretation

1. **What is this data about?**

   The dataset represents Instamart's completed customer orders. It contains information about order amounts, quantities, discounts and delivery times. It helps understand customer purchasing behaviour and business revenue.

2. **What do its distributions and statistics look like?**

   The average order amount is INR 269.91, while the median is INR 135.00. The skewness is approximately 4.61, indicating a positively (right) skewed distribution. The first quartile (Q1) is INR 64.80 and the third quartile (Q3) is INR 297.00.

3. **Are there outliers or anomalies?**

   Using the 1.5 × IQR method, the upper fence is INR 645.30. There are 897 outliers above the upper fence and no outliers below the lower fence. This indicates that some orders have unusually high amounts.

4. **What relationships exist between variables?**

   The correlation matrix shows that order amount and discount percentage have the strongest correlation in absolute value, approximately -0.095. Order amount and quantity have a correlation of +0.087. Both relationships are very weak. Correlation of this size does not establish causation.

# ---------- PART A IS DONE ----------